# 45｜从零实现 ELECTRA：小 Generator、替换采样与 Replaced-Token Detection

本 Notebook 不调用 transformers、nn.Transformer 或 nn.MultiheadAttention。我们手写双向 encoder attention、generator MLM、从词表分布采样 replacement、discriminator 的 replaced-token detection（RTD）以及联合 loss。

ELECTRA 的关键边界是：被选为 MLM 的位置不一定真的被替换。若 generator 恰好采样回原 token，RTD 标签必须是 0；标签应由 original_ids 与 corrupted_ids 的逐位置比较得到。special token 和 padding 不参加 MLM，也不参加本教学版 RTD loss。

> 实验边界：CPU、离线、单线程、小合成数据；受控 loss 下降只验证预训练数据流，不能代表下游迁移或真实语料泛化。

## 1. ELECTRA 数据流与张量合同

1. 从每行普通 token 中选择 MLM 位置，把输入改成 MASK，未选 label 设为 -100。
2. generator 输出 [B,T,V]，只在被选位置计算 token CE。
3. 从 generator 的普通词分布采样 replacement，并写回原序列。
4. discriminator 读取 corrupted sequence，逐 token 输出一个 logit。
5. RTD 标签是 corrupted!=original；special/padding 的标签为 -100 并排除 BCE。

attention 隐状态为 [B,T,D]、拆头 [B,H,T,d_h]、权重 [B,H,T,T]。双向 encoder 没有 causal mask，复杂度主项为 O(B·H·T²)。

In [ ]:
import copy
import hashlib
import json
import math
import random
import warnings
from types import MappingProxyType

warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)

import torch
from torch import nn
import torch.nn.functional as F

SEED45 = 4507
random.seed(SEED45)
torch.manual_seed(SEED45)
torch.set_num_threads(1)
DEVICE45 = torch.device("cpu")

PAD45, CLS45, SEP45, MASK45, UNK45 = 0, 1, 2, 3, 4
VOCAB45 = [
    "<pad>", "<cls>", "<sep>", "<mask>", "<unk>",
    "北京", "上海", "天气", "晴朗", "今天", "明天", "模型", "学习",
    "文本", "图像", "检索", "知识", "系统", "数据", "分类", "生成",
    "语言", "理解", "搜索",
]
TOKEN_TO_ID45 = {token: index for index, token in enumerate(VOCAB45)}
SPECIAL45 = {PAD45, CLS45, SEP45, MASK45, UNK45}
ORDINARY45 = list(range(max(SPECIAL45) + 1, len(VOCAB45)))

assert DEVICE45.type == "cpu"
assert torch.get_num_threads() == 1
assert len(VOCAB45) == len(TOKEN_TO_ID45)
assert set(ORDINARY45).isdisjoint(SPECIAL45)
assert ORDINARY45[0] == 5 and ORDINARY45[-1] == len(VOCAB45) - 1
print({"torch": torch.__version__, "device": str(DEVICE45), "vocab": len(VOCAB45)})

## 2. 先切分原始记录，再选择 MLM 位置

数据切分的单位应是原始文档或语义组，而不是 masking 后的样本。若先生成多个 mask 版本再随机切分，同一原句很可能同时出现在 train/validation，导致验证泄漏。

下面每行使用局部 Generator 选择普通 token；只要存在候选且 probability>0，就保证至少一个监督位置。局部 RNG 让同 seed 可重现且不污染全局随机状态。

selected 位置采用显式 **80/10/10** generator 输入策略：80% 写 MASK、10% 写随机普通词、10% 保持原词。位置选择、策略动作和随机普通词分别使用由基础 seed 派生的三个独立 `torch.Generator`；随机词即使碰巧等于原词，策略仍记录为 random。`policy` 用 `0/1/2` 表示三种动作，未选择位置为 -1，便于直接审计而不是从最终 token 猜动作。


In [ ]:
RAW_RECORDS45 = [
    {"id": "e0", "tokens": [5, 9, 7, 8, 17]},
    {"id": "e1", "tokens": [6, 10, 7, 8, 17]},
    {"id": "e2", "tokens": [11, 12, 13, 23, 17]},
    {"id": "e3", "tokens": [14, 15, 18, 16, 17]},
    {"id": "e4", "tokens": [19, 13, 11, 12, 20]},
    {"id": "e5", "tokens": [21, 22, 13, 20, 23]},
]
TRAIN_IDS45, VALID_IDS45 = ["e0", "e1", "e2", "e3"], ["e4", "e5"]
POLICY_SEED_OFFSET45 = 1_000_003
RANDOM_TOKEN_SEED_OFFSET45 = 2_000_003
REPLACEMENT_SEED_OFFSET45 = 10_000


def collate_records45(records):
    sequences = [[CLS45, *record["tokens"], SEP45] for record in records]
    max_length = max(len(sequence) for sequence in sequences)
    ids = torch.full((len(sequences), max_length), PAD45, dtype=torch.long)
    mask = torch.zeros_like(ids, dtype=torch.bool)
    for row, sequence in enumerate(sequences):
        ids[row, :len(sequence)] = torch.tensor(sequence)
        mask[row, :len(sequence)] = True
    return ids, mask


def choose_mlm45(input_ids, attention_mask, probability, seed):
    if input_ids.dtype != torch.long or input_ids.ndim != 2:
        raise ValueError("input_ids 必须是二维 long")
    if attention_mask.shape != input_ids.shape or attention_mask.dtype != torch.bool:
        raise ValueError("attention_mask 合同错误")
    if not 0.0 <= probability <= 1.0:
        raise ValueError("probability 必须在 [0,1]")
    eligible = attention_mask.clone()
    for special in SPECIAL45:
        eligible &= input_ids.ne(special)
    selected = torch.zeros_like(eligible)
    selection_generator = torch.Generator().manual_seed(seed)
    for row in range(input_ids.shape[0]):
        candidates = eligible[row].nonzero(as_tuple=False).flatten()
        if candidates.numel() and probability > 0:
            count = max(1, int(round(candidates.numel() * probability)))
            permutation = torch.randperm(candidates.numel(), generator=selection_generator)
            selected[row, candidates[permutation[:count]]] = True

    labels = input_ids.masked_fill(~selected, -100)
    generator_input = input_ids.clone()
    policy = torch.full_like(input_ids, -1)  # -1未选；0 MASK；1随机普通词；2保持原词
    selected_positions = selected.nonzero(as_tuple=False)
    if selected_positions.numel():
        policy_generator = torch.Generator().manual_seed(seed + POLICY_SEED_OFFSET45)
        random_token_generator = torch.Generator().manual_seed(seed + RANDOM_TOKEN_SEED_OFFSET45)
        draws = torch.rand(selected_positions.shape[0], generator=policy_generator)
        actions = torch.where(draws < 0.8, 0, torch.where(draws < 0.9, 1, 2)).long()
        policy[selected] = actions
        mask_positions = selected_positions[actions == 0]
        random_positions = selected_positions[actions == 1]
        if mask_positions.numel():
            generator_input[mask_positions[:, 0], mask_positions[:, 1]] = MASK45
        if random_positions.numel():
            sampled = torch.randint(
                0, len(ORDINARY45), (random_positions.shape[0],), generator=random_token_generator
            )
            ordinary = torch.tensor(ORDINARY45)
            generator_input[random_positions[:, 0], random_positions[:, 1]] = ordinary[sampled]
    return generator_input, labels, selected, policy


train_records45 = [record for record in RAW_RECORDS45 if record["id"] in TRAIN_IDS45]
train_ids45, train_attention45 = collate_records45(train_records45)
generator_input45, mlm_labels45, selected45, policy45 = choose_mlm45(
    train_ids45, train_attention45, probability=0.4, seed=SEED45
)
generator_input_again45, labels_again45, selected_again45, policy_again45 = choose_mlm45(
    train_ids45, train_attention45, probability=0.4, seed=SEED45
)

assert generator_input45.equal(generator_input_again45)
assert mlm_labels45.equal(labels_again45) and selected45.equal(selected_again45)
assert policy45.equal(policy_again45)
assert selected45.sum(dim=1).eq(2).all()
assert policy45[~selected45].eq(-1).all()
assert set(policy45[selected45].tolist()).issubset({0, 1, 2})
assert generator_input45[(policy45 == 0)].eq(MASK45).all()
assert mlm_labels45[~selected45].eq(-100).all()
for special45 in SPECIAL45:
    assert not bool((selected45 & train_ids45.eq(special45)).any())

# 大样本只检验策略概率，不把“随机词碰巧等于原词”误算成 unchanged action。
policy_probe_ids45 = torch.tensor(ORDINARY45 * 100).reshape(100, len(ORDINARY45))
policy_probe_mask45 = torch.ones_like(policy_probe_ids45, dtype=torch.bool)
_, _, policy_probe_selected45, policy_probe45 = choose_mlm45(
    policy_probe_ids45, policy_probe_mask45, probability=1.0, seed=SEED45 + 333
)
policy_counts45 = torch.bincount(policy_probe45[policy_probe_selected45], minlength=3).float()
policy_rates45 = policy_counts45 / policy_counts45.sum()
assert 0.76 < float(policy_rates45[0]) < 0.84
assert 0.07 < float(policy_rates45[1]) < 0.13
assert 0.07 < float(policy_rates45[2]) < 0.13
assert set(generator_input45[(policy45 == 1)].tolist()).issubset(set(ORDINARY45))
assert set(TRAIN_IDS45).isdisjoint(VALID_IDS45)
assert set(TRAIN_IDS45) | set(VALID_IDS45) == {record["id"] for record in RAW_RECORDS45}

## 3. 手写双向 self-attention

Q、K、V 线性投影后拆头，score=QKᵀ/sqrt(d_h)。这里只屏蔽 padding key，没有 causal 上三角，因此右侧 token 可以影响左侧表示。softmax 后把 padding query 的权重与输出显式归零，避免 output projection bias 重新引入非零值。

In [ ]:
class ManualSelfAttention45(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        if dim % heads:
            raise ValueError("dim 必须整除 heads")
        self.dim, self.heads, self.head_dim = dim, heads, dim // heads
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim)

    def _split(self, x):
        batch, length, _ = x.shape
        return x.view(batch, length, self.heads, self.head_dim).transpose(1, 2)

    def forward(self, x, attention_mask):
        if x.ndim != 3 or x.shape[-1] != self.dim:
            raise ValueError("attention 输入形状错误")
        if attention_mask.shape != x.shape[:2] or attention_mask.dtype != torch.bool:
            raise ValueError("attention_mask 合同错误")
        if not bool(attention_mask.any(dim=1).all()):
            raise ValueError("每行至少需要一个有效 token")
        q, k, v = self._split(self.q_proj(x)), self._split(self.k_proj(x)), self._split(self.v_proj(x))
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        visible = attention_mask[:, None, None, :]
        weights = torch.softmax(scores.masked_fill(~visible, -torch.inf), dim=-1)
        weights = weights * attention_mask[:, None, :, None]
        context = torch.matmul(weights, v).transpose(1, 2).contiguous().view_as(x)
        output = self.out_proj(context) * attention_mask.unsqueeze(-1)
        return output, weights


attention_probe45 = ManualSelfAttention45(4, 2)
with torch.no_grad():
    for layer45 in [
        attention_probe45.q_proj, attention_probe45.k_proj,
        attention_probe45.v_proj, attention_probe45.out_proj,
    ]:
        layer45.weight.copy_(torch.eye(4))
        layer45.bias.zero_()
x_probe45 = torch.tensor([[[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0]]])
mask_probe45 = torch.ones(1, 2, dtype=torch.bool)
output_probe45, weights_probe45 = attention_probe45(x_probe45, mask_probe45)
expected_scores45 = torch.tensor([[1.0, 0.0], [0.0, 1.0]]) / math.sqrt(2.0)
assert torch.allclose(weights_probe45[0, 0], torch.softmax(expected_scores45, -1), atol=1e-7)
assert torch.allclose(weights_probe45[0, 1], torch.full((2, 2), 0.5), atol=1e-7)

padded_mask45 = torch.tensor([[True, False]])
padded_output45, padded_weights45 = attention_probe45(x_probe45, padded_mask45)
assert padded_output45[0, 1].abs().max().item() == 0.0
assert padded_weights45[0, :, 1].abs().max().item() == 0.0
assert padded_weights45[0, :, 0, 1].abs().max().item() == 0.0

## 4. Generator 与 Discriminator 的容量和共享策略

论文中 generator 通常比 discriminator 小，但可以共享 token embedding。为让教学代码简洁，两者 hidden dim 相同、generator 一层、discriminator 两层，并共享同一个 embedding Parameter；encoder block、position embedding 与 attention 参数各自独立。

共享能减少参数并让两种目标共同更新词表示，但也耦合了优化。生产实验应把共享开关、容量比和 loss 权重作为显式配置，不能只在代码中默默决定。

In [ ]:
class EncoderBlock45(nn.Module):
    def __init__(self, dim, heads, hidden_dim):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attention = ManualSelfAttention45(dim, heads)
        self.norm2 = nn.LayerNorm(dim)
        self.ff1 = nn.Linear(dim, hidden_dim)
        self.ff2 = nn.Linear(hidden_dim, dim)

    def forward(self, x, attention_mask):
        update, _ = self.attention(self.norm1(x), attention_mask)
        x = (x + update) * attention_mask.unsqueeze(-1)
        update = self.ff2(F.gelu(self.ff1(self.norm2(x))))
        return (x + update) * attention_mask.unsqueeze(-1)


class TinyEncoder45(nn.Module):
    def __init__(self, vocab_size, dim, heads, hidden_dim, layers, max_length, embedding=None):
        super().__init__()
        self.token_embedding = embedding if embedding is not None else nn.Embedding(
            vocab_size, dim, padding_idx=PAD45
        )
        if self.token_embedding.embedding_dim != dim:
            raise ValueError("共享 embedding dim 不匹配")
        self.position_embedding = nn.Embedding(max_length, dim)
        self.blocks = nn.ModuleList(
            [EncoderBlock45(dim, heads, hidden_dim) for _ in range(layers)]
        )
        self.final_norm = nn.LayerNorm(dim)
        self.max_length = max_length

    def forward(self, input_ids, attention_mask):
        if input_ids.dtype != torch.long or input_ids.ndim != 2:
            raise ValueError("input_ids 必须是二维 long")
        if attention_mask.shape != input_ids.shape or attention_mask.dtype != torch.bool:
            raise ValueError("attention_mask 合同错误")
        if input_ids.shape[1] > self.max_length:
            raise ValueError("序列超过 position embedding 上限")
        if not bool(attention_mask.any(dim=1).all()):
            raise ValueError("每行至少一个有效 token")
        positions = torch.arange(input_ids.shape[1], device=input_ids.device)
        hidden = self.token_embedding(input_ids) + self.position_embedding(positions)[None]
        hidden = hidden * attention_mask.unsqueeze(-1)
        for block in self.blocks:
            hidden = block(hidden, attention_mask)
        return self.final_norm(hidden) * attention_mask.unsqueeze(-1)


class ElectraPretrainer45(nn.Module):
    def __init__(
        self, vocab_size, dim=16, heads=4, hidden_dim=32,
        generator_layers=1, discriminator_layers=2, max_length=16,
        share_embeddings=True,
    ):
        super().__init__()
        self.config = {
            "vocab_size": vocab_size, "dim": dim, "heads": heads,
            "hidden_dim": hidden_dim, "generator_layers": generator_layers,
            "discriminator_layers": discriminator_layers,
            "max_length": max_length, "share_embeddings": share_embeddings,
        }
        shared = nn.Embedding(vocab_size, dim, padding_idx=PAD45) if share_embeddings else None
        self.generator = TinyEncoder45(
            vocab_size, dim, heads, hidden_dim, generator_layers, max_length, shared
        )
        discriminator_embedding = shared if share_embeddings else None
        self.discriminator = TinyEncoder45(
            vocab_size, dim, heads, hidden_dim, discriminator_layers,
            max_length, discriminator_embedding,
        )
        self.generator_bias = nn.Parameter(torch.zeros(vocab_size))
        self.discriminator_head = nn.Linear(dim, 1)

    def generator_logits(self, masked_ids, attention_mask):
        hidden = self.generator(masked_ids, attention_mask)
        return F.linear(hidden, self.generator.token_embedding.weight, self.generator_bias)

    def discriminator_logits(self, corrupted_ids, attention_mask):
        hidden = self.discriminator(corrupted_ids, attention_mask)
        return self.discriminator_head(hidden).squeeze(-1)

    def forward(self, input_ids, attention_mask, head):
        if head == "generator":
            return self.generator_logits(input_ids, attention_mask)
        if head == "discriminator":
            return self.discriminator_logits(input_ids, attention_mask)
        raise ValueError("head 必须是 generator 或 discriminator")


torch.manual_seed(SEED45)
model45 = ElectraPretrainer45(len(VOCAB45))
generator_logits_probe45 = model45.generator_logits(generator_input45, train_attention45)
discriminator_logits_probe45 = model45.discriminator_logits(train_ids45, train_attention45)
assert generator_logits_probe45.shape == (*train_ids45.shape, len(VOCAB45))
assert discriminator_logits_probe45.shape == train_ids45.shape
assert model45.generator.token_embedding.weight is model45.discriminator.token_embedding.weight
assert model45.generator.token_embedding.weight.data_ptr() == model45.discriminator.token_embedding.weight.data_ptr()

separate_model45 = ElectraPretrainer45(len(VOCAB45), share_embeddings=False)
assert separate_model45.generator.token_embedding.weight is not separate_model45.discriminator.token_embedding.weight
assert sum(p.numel() for p in model45.generator.parameters()) < sum(
    p.numel() for p in model45.discriminator.parameters()
)

## 5. 从 generator 分布采样，而不是把 selected 直接当作 replaced

只在 selected 位置采样，并禁止生成 PAD/CLS/SEP/MASK/UNK。corrupted sequence 从 original 复制，再写入 sampled token；RTD truth 随后统一通过 corrupted!=original 计算。

若采样恰好等于原 token，该位置虽然产生 generator loss，但 discriminator 标签为 0。这是 ELECTRA 数据流中最重要的边界条件之一。

In [ ]:
def sample_replacements45(generator_logits, original_ids, selected, seed, temperature=1.0):
    if generator_logits.shape[:2] != original_ids.shape or selected.shape != original_ids.shape:
        raise ValueError("replacement 形状合同错误")
    if selected.dtype != torch.bool or temperature <= 0:
        raise ValueError("selected/temperature 合同错误")
    if not torch.isfinite(generator_logits).all():
        raise ValueError("generator logits 必须有限")
    corrupted = original_ids.clone()
    local_generator = torch.Generator().manual_seed(seed)
    ordinary_index = torch.tensor(ORDINARY45, device=generator_logits.device)
    for row, position in selected.nonzero(as_tuple=False).tolist():
        logits = generator_logits[row, position, ordinary_index] / temperature
        probabilities = torch.softmax(logits, dim=-1)
        sampled_local = torch.multinomial(probabilities.cpu(), 1, generator=local_generator).item()
        corrupted[row, position] = ordinary_index[sampled_local]
    return corrupted


def rtd_labels45(original_ids, corrupted_ids, attention_mask):
    if original_ids.shape != corrupted_ids.shape or original_ids.shape != attention_mask.shape:
        raise ValueError("RTD 输入形状错误")
    eligible = attention_mask.clone()
    for special in SPECIAL45:
        eligible &= original_ids.ne(special)
    labels = corrupted_ids.ne(original_ids).long()
    return labels.masked_fill(~eligible, -100), eligible


selected_edge45 = torch.zeros(1, 4, dtype=torch.bool)
selected_edge45[0, 1] = True
original_edge45 = torch.tensor([[CLS45, 7, 8, SEP45]])
same_logits45 = torch.full((1, 4, len(VOCAB45)), -100.0)
same_logits45[0, 1, 7] = 100.0
same_corrupted45 = sample_replacements45(
    same_logits45, original_edge45, selected_edge45, seed=1
)
same_labels45, same_eligible45 = rtd_labels45(
    original_edge45, same_corrupted45, torch.ones_like(original_edge45, dtype=torch.bool)
)
assert same_corrupted45[0, 1].item() == original_edge45[0, 1].item()
assert same_labels45[0, 1].item() == 0
assert selected_edge45[0, 1] and same_labels45[0, 1].item() != 1

different_logits45 = torch.full((1, 4, len(VOCAB45)), -100.0)
different_logits45[0, 1, 9] = 100.0
different_corrupted45 = sample_replacements45(
    different_logits45, original_edge45, selected_edge45, seed=1
)
different_labels45, _ = rtd_labels45(
    original_edge45, different_corrupted45,
    torch.ones_like(original_edge45, dtype=torch.bool),
)
assert different_corrupted45[0, 1].item() == 9
assert different_labels45[0, 1].item() == 1
assert same_labels45[0, 0].item() == -100 and same_labels45[0, 3].item() == -100
assert same_eligible45.tolist() == [[False, True, True, False]]

## 6. Generator CE、Discriminator BCE 与联合权重

generator loss 只平均 selected token；discriminator loss 平均所有普通、非 padding token。两者监督密度差异很大，因此联合目标显式写成 generator_loss + lambda·discriminator_loss。

lambda 是优化超参数，不应把两个未归一化 loss 直接相加。下面用小张量逐项重算，防止 ignore mask 或平均分母写错。

In [ ]:
def generator_loss45(logits, mlm_labels):
    supervised = mlm_labels.ne(-100)
    if logits.shape[:2] != mlm_labels.shape or not bool(supervised.any()):
        raise ValueError("generator loss 缺少合法监督")
    return F.cross_entropy(
        logits.reshape(-1, logits.shape[-1]), mlm_labels.reshape(-1), ignore_index=-100
    )


def discriminator_loss45(logits, labels):
    supervised = labels.ne(-100)
    if logits.shape != labels.shape or not bool(supervised.any()):
        raise ValueError("discriminator loss 缺少合法监督")
    return F.binary_cross_entropy_with_logits(
        logits[supervised], labels[supervised].float()
    )


def electra_objective45(
    model, original_ids, attention_mask, probability, seed,
    discriminator_weight=5.0,
):
    generator_input_ids, mlm_labels, selected, policy = choose_mlm45(
        original_ids, attention_mask, probability, seed
    )
    generator_logits = model.generator_logits(generator_input_ids, attention_mask)
    corrupted_ids = sample_replacements45(
        generator_logits.detach(), original_ids, selected, seed + REPLACEMENT_SEED_OFFSET45
    )
    labels, eligible = rtd_labels45(original_ids, corrupted_ids, attention_mask)
    discriminator_logits = model.discriminator_logits(corrupted_ids, attention_mask)
    generator_loss = generator_loss45(generator_logits, mlm_labels)
    discriminator_loss = discriminator_loss45(discriminator_logits, labels)
    total = generator_loss + discriminator_weight * discriminator_loss
    return {
        "total": total, "generator_loss": generator_loss,
        "discriminator_loss": discriminator_loss,
        "generator_input_ids": generator_input_ids, "corrupted_ids": corrupted_ids,
        "mlm_labels": mlm_labels, "rtd_labels": labels,
        "selected": selected, "policy": policy, "eligible": eligible,
    }


tiny_generator_logits45 = torch.tensor([[[2.0, 0.0], [0.0, 1.0], [8.0, -8.0]]])
tiny_generator_labels45 = torch.tensor([[0, 1, -100]])
actual_generator_loss45 = generator_loss45(tiny_generator_logits45, tiny_generator_labels45)
manual_generator_loss45 = (
    -F.log_softmax(tiny_generator_logits45[0, 0], -1)[0]
    -F.log_softmax(tiny_generator_logits45[0, 1], -1)[1]
) / 2
assert torch.allclose(actual_generator_loss45, manual_generator_loss45, atol=1e-7)

tiny_discriminator_logits45 = torch.tensor([[0.0, math.log(3.0), -9.0]])
tiny_discriminator_labels45 = torch.tensor([[0, 1, -100]])
actual_discriminator_loss45 = discriminator_loss45(
    tiny_discriminator_logits45, tiny_discriminator_labels45
)
manual_discriminator_loss45 = (
    F.softplus(tiny_discriminator_logits45[0, 0])
    + F.softplus(-tiny_discriminator_logits45[0, 1])
) / 2
assert torch.allclose(actual_discriminator_loss45, manual_discriminator_loss45, atol=1e-7)

objective_probe45 = electra_objective45(
    model45, train_ids45, train_attention45, 0.4, SEED45
)
assert torch.allclose(
    objective_probe45["total"],
    objective_probe45["generator_loss"] + 5.0 * objective_probe45["discriminator_loss"],
)
assert objective_probe45["rtd_labels"][~objective_probe45["eligible"]].eq(-100).all()
assert objective_probe45["corrupted_ids"][~objective_probe45["selected"]].equal(
    train_ids45[~objective_probe45["selected"]]
)

## 7. 受控联合预训练

每一步改变局部 seed，让模型看到不同 MLM 位置与采样结果；不使用验证记录更新参数。最终在一组固定 seed 上比较 generator loss，并检查梯度有限。RTD 的正例比例会随 generator 变强而变化，所以 discriminator accuracy 不能脱离类别比例直接解释。

In [ ]:
torch.manual_seed(SEED45)
model45 = ElectraPretrainer45(
    len(VOCAB45), dim=16, heads=4, hidden_dim=32,
    generator_layers=1, discriminator_layers=2, max_length=16,
    share_embeddings=True,
)
optimizer45 = torch.optim.Adam(model45.parameters(), lr=0.015)

def evaluate_generator45(model, seeds):
    values = []
    model.eval()
    with torch.no_grad():
        for seed45 in seeds:
            result45 = electra_objective45(
                model, train_ids45, train_attention45, 0.4, seed45
            )
            values.append(float(result45["generator_loss"]))
    return sum(values) / len(values)


evaluation_seeds45 = [SEED45 + offset for offset in range(4)]
initial_generator_loss45 = evaluate_generator45(model45, evaluation_seeds45)
history45 = []
model45.train()
for step in range(45):
    optimizer45.zero_grad(set_to_none=True)
    result45 = electra_objective45(
        model45, train_ids45, train_attention45,
        probability=0.4, seed=SEED45 + (step % 12),
        discriminator_weight=5.0,
    )
    result45["total"].backward()
    torch.nn.utils.clip_grad_norm_(model45.parameters(), 1.0)
    optimizer45.step()
    if step in {0, 9, 29, 44}:
        history45.append((
            step, float(result45["generator_loss"].detach()),
            float(result45["discriminator_loss"].detach()),
        ))

final_generator_loss45 = evaluate_generator45(model45, evaluation_seeds45)
model45.eval()
fixed_result45 = electra_objective45(
    model45, train_ids45, train_attention45, 0.4, SEED45
)
with torch.no_grad():
    valid_records45 = [record for record in RAW_RECORDS45 if record["id"] in VALID_IDS45]
    valid_ids45, valid_attention45 = collate_records45(valid_records45)
    valid_result45 = electra_objective45(
        model45, valid_ids45, valid_attention45, 0.4, SEED45 + 99
    )

assert math.isfinite(initial_generator_loss45) and math.isfinite(final_generator_loss45)
assert final_generator_loss45 < initial_generator_loss45 * 0.55
assert torch.isfinite(fixed_result45["total"])
assert torch.isfinite(valid_result45["total"])
assert fixed_result45["eligible"].sum().item() == 4 * 5
assert valid_result45["eligible"].sum().item() == 2 * 5
positive_rate45 = fixed_result45["rtd_labels"][fixed_result45["eligible"]].float().mean().item()
assert 0.0 <= positive_rate45 <= 1.0
print({
    "initial_generator_loss": round(initial_generator_loss45, 4),
    "final_generator_loss": round(final_generator_loss45, 4),
    "rtd_positive_rate": round(positive_rate45, 3),
    "trace": history45,
})

## 8. 双向性、padding 不变性与梯度归属

ELECTRA encoder 是双向的：改变一个未来普通 token，左侧表示通常应变化；但追加 masked padding 不应改变原序列表示。共享 embedding 必须同时从 generator CE 和 discriminator BCE 收到梯度。

这些结构 oracle 与任务指标互补：前者检查实现合同，后者才回答模型是否在目标分布有效。

联合 backward 不能证明两条目标各自连通：其中一条断梯度时，另一条仍可能让共享 embedding 的总梯度非零。因此下面分别只反传 generator CE 与 discriminator BCE，既检查共享 embedding 的非零梯度，也检查非对应的 discriminator/generator 专属 head 保持 `grad is None`。


In [ ]:
model45.eval()
bidirectional_a45 = torch.tensor([[CLS45, 5, 9, 7, SEP45]])
bidirectional_b45 = bidirectional_a45.clone()
bidirectional_b45[0, 3] = 18
bidirectional_mask45 = torch.ones_like(bidirectional_a45, dtype=torch.bool)
with torch.no_grad():
    hidden_a45 = model45.discriminator(bidirectional_a45, bidirectional_mask45)
    hidden_b45 = model45.discriminator(bidirectional_b45, bidirectional_mask45)
assert not torch.allclose(hidden_a45[:, 1], hidden_b45[:, 1])

extended_ids45 = torch.cat([bidirectional_a45, torch.tensor([[18, 19]])], dim=1)
extended_mask45 = torch.cat(
    [bidirectional_mask45, torch.zeros(1, 2, dtype=torch.bool)], dim=1
)
with torch.no_grad():
    hidden_extended45 = model45.discriminator(extended_ids45, extended_mask45)
assert torch.allclose(hidden_a45, hidden_extended45[:, :5], atol=2e-5)
assert hidden_extended45[:, 5:].abs().max().item() == 0.0

model45.zero_grad(set_to_none=True)
gradient_result45 = electra_objective45(
    model45, train_ids45, train_attention45, 0.4, SEED45 + 123
)
gradient_result45["total"].backward()
gradients45 = [parameter.grad for parameter in model45.parameters() if parameter.grad is not None]
assert gradients45 and all(torch.isfinite(gradient).all() for gradient in gradients45)
shared_gradient45 = model45.generator.token_embedding.weight.grad
assert shared_gradient45 is model45.discriminator.token_embedding.weight.grad
assert shared_gradient45.abs().sum().item() > 0
assert model45.discriminator_head.weight.grad.abs().sum().item() > 0

# 两条 loss 分开反传：共享 embedding 必须分别收到梯度，非对应 head 不得收到梯度。
model45.zero_grad(set_to_none=True)
generator_only_logits45 = model45.generator_logits(generator_input45, train_attention45)
generator_loss45(generator_only_logits45, mlm_labels45).backward()
generator_shared_grad45 = model45.generator.token_embedding.weight.grad
assert generator_shared_grad45 is not None and generator_shared_grad45.abs().sum().item() > 0
assert model45.discriminator_head.weight.grad is None
assert model45.discriminator.blocks[0].ff1.weight.grad is None

model45.zero_grad(set_to_none=True)
with torch.no_grad():
    discriminator_generator_logits45 = model45.generator_logits(generator_input45, train_attention45)
    discriminator_corrupted45 = sample_replacements45(
        discriminator_generator_logits45, train_ids45, selected45,
        SEED45 + REPLACEMENT_SEED_OFFSET45,
    )
    discriminator_labels45, _ = rtd_labels45(
        train_ids45, discriminator_corrupted45, train_attention45,
    )
discriminator_only_logits45 = model45.discriminator_logits(discriminator_corrupted45, train_attention45)
discriminator_loss45(discriminator_only_logits45, discriminator_labels45).backward()
discriminator_shared_grad45 = model45.generator.token_embedding.weight.grad
assert discriminator_shared_grad45 is not None and discriminator_shared_grad45.abs().sum().item() > 0
assert model45.generator_bias.grad is None
assert model45.generator.blocks[0].ff1.weight.grad is None


# 两条 loss 分开反传：共享 embedding 必须分别收到梯度，非对应 head 不得收到梯度。
model45.zero_grad(set_to_none=True)
generator_only_logits45 = model45.generator_logits(generator_input45, train_attention45)
generator_loss45(generator_only_logits45, mlm_labels45).backward()
generator_shared_grad45 = model45.generator.token_embedding.weight.grad
assert generator_shared_grad45 is not None and generator_shared_grad45.abs().sum().item() > 0
assert model45.discriminator_head.weight.grad is None
assert model45.discriminator.blocks[0].ff1.weight.grad is None

model45.zero_grad(set_to_none=True)
with torch.no_grad():
    discriminator_generator_logits45 = model45.generator_logits(generator_input45, train_attention45)
    discriminator_corrupted45 = sample_replacements45(
        discriminator_generator_logits45, train_ids45, selected45,
        SEED45 + REPLACEMENT_SEED_OFFSET45,
    )
    discriminator_labels45, _ = rtd_labels45(
        train_ids45, discriminator_corrupted45, train_attention45,
    )
discriminator_only_logits45 = model45.discriminator_logits(discriminator_corrupted45, train_attention45)
discriminator_loss45(discriminator_only_logits45, discriminator_labels45).backward()
discriminator_shared_grad45 = model45.generator.token_embedding.weight.grad
assert discriminator_shared_grad45 is not None and discriminator_shared_grad45.abs().sum().item() > 0
assert model45.generator_bias.grad is None
assert model45.generator.blocks[0].ff1.weight.grad is None


## 9. 可信发布：完整绑定 replacement 语义

manifest 不只保存模型尺寸，还绑定完整词表、原始记录、互斥 split、special 排除规则、mask 概率、采样温度、真假标签公式、loss 权重和训练 recipe。否则相同权重配上不同 tokenizer 或把 selected 当 replaced，都可能静默改变模型语义。

canonical state digest 包含 key、dtype、shape、bytes；外部只读 publisher registry 保存整个 package 的预期指纹。包内 hash 可做完整性提示，但攻击者整体替换并重算它们仍必须被 registry 拒绝。

In [ ]:
def canonical_json45(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode("utf-8")


def canonical_state_digest45(state):
    digest = hashlib.sha256()
    for key in sorted(state):
        tensor = state[key].detach().cpu().contiguous()
        digest.update(canonical_json45({
            "key": key, "dtype": str(tensor.dtype), "shape": list(tensor.shape)
        }))
        digest.update(tensor.numpy().tobytes(order="C"))
    return digest.hexdigest()


def package_fingerprint45(package):
    digest = hashlib.sha256()
    digest.update(canonical_json45(package["manifest"]))
    digest.update(canonical_state_digest45(package["state"]).encode("ascii"))
    return digest.hexdigest()


manifest45 = {
    "subject": "electra-pretraining-demo@1",
    "architecture": copy.deepcopy(model45.config),
    "vocab": list(VOCAB45),
    "special_ids": {
        "pad": PAD45, "cls": CLS45, "sep": SEP45,
        "mask": MASK45, "unk": UNK45,
    },
    "dataset": copy.deepcopy(RAW_RECORDS45),
    "split": {"train": list(TRAIN_IDS45), "validation": list(VALID_IDS45)},
    "preprocess": {
        "tokenizer": "frozen-token-id-v1", "padding": "right",
        "mlm_probability": 0.4, "replacement_temperature": 1.0,
        "generator_input_policy": {"mask": 0.8, "random_ordinary": 0.1, "unchanged": 0.1},
        "selection_count": "round(candidate_count*probability)-minimum-one-per-nonempty-row",
        "seed_derivation": {
            "selection": 0, "policy": POLICY_SEED_OFFSET45,
            "random_token": RANDOM_TOKEN_SEED_OFFSET45,
            "generator_replacement": REPLACEMENT_SEED_OFFSET45,
        },
        "sample_domain": list(ORDINARY45),
        "rtd_truth": "corrupted_id != original_id",
        "excluded_from_losses": sorted(SPECIAL45),
    },
    "recipe": {
        "seed": SEED45, "optimizer": "Adam", "learning_rate": 0.015,
        "steps": 45, "gradient_clip": 1.0,
        "generator_loss": "selected-token-cross-entropy",
        "discriminator_loss": "eligible-token-bce-with-logits",
        "discriminator_weight": 5.0,
    },
}
state45 = {key: value.detach().cpu().clone() for key, value in model45.state_dict().items()}
package45 = {
    "manifest": manifest45,
    "state": state45,
    "internal": {
        "manifest_digest": hashlib.sha256(canonical_json45(manifest45)).hexdigest(),
        "state_digest": canonical_state_digest45(state45),
    },
}
subject45 = manifest45["subject"]
PUBLISHER_REGISTRY45 = MappingProxyType({subject45: package_fingerprint45(package45)})


def load_published_electra45(package, subject):
    if subject not in PUBLISHER_REGISTRY45:
        raise ValueError("未知发布 subject")
    if package_fingerprint45(package) != PUBLISHER_REGISTRY45[subject]:
        raise ValueError("publisher registry 指纹不匹配")
    manifest = package["manifest"]
    if manifest["subject"] != subject:
        raise ValueError("subject 不匹配")
    if package["internal"]["manifest_digest"] != hashlib.sha256(canonical_json45(manifest)).hexdigest():
        raise ValueError("manifest 内部摘要不匹配")
    if package["internal"]["state_digest"] != canonical_state_digest45(package["state"]):
        raise ValueError("state 内部摘要不匹配")
    if manifest["vocab"] != VOCAB45 or manifest["special_ids"]["mask"] != MASK45:
        raise ValueError("词表或 special id 不匹配")
    train_ids = set(manifest["split"]["train"])
    validation_ids = set(manifest["split"]["validation"])
    data_ids = {record["id"] for record in manifest["dataset"]}
    if train_ids & validation_ids or train_ids | validation_ids != data_ids:
        raise ValueError("split 非互斥或未覆盖")
    expected_policy45 = {"mask": 0.8, "random_ordinary": 0.1, "unchanged": 0.1}
    expected_seed_derivation45 = {
        "selection": 0, "policy": POLICY_SEED_OFFSET45,
        "random_token": RANDOM_TOKEN_SEED_OFFSET45,
        "generator_replacement": REPLACEMENT_SEED_OFFSET45,
    }
    if manifest["preprocess"]["generator_input_policy"] != expected_policy45:
        raise ValueError("generator 80/10/10 策略不匹配")
    if manifest["preprocess"]["selection_count"] != "round(candidate_count*probability)-minimum-one-per-nonempty-row":
        raise ValueError("MLM selection/min-one 策略不匹配")
    if manifest["preprocess"]["seed_derivation"] != expected_seed_derivation45:
        raise ValueError("随机流派生规则不匹配")
    expected_policy45 = {"mask": 0.8, "random_ordinary": 0.1, "unchanged": 0.1}
    expected_seed_derivation45 = {
        "selection": 0, "policy": POLICY_SEED_OFFSET45,
        "random_token": RANDOM_TOKEN_SEED_OFFSET45,
        "generator_replacement": REPLACEMENT_SEED_OFFSET45,
    }
    if manifest["preprocess"]["generator_input_policy"] != expected_policy45:
        raise ValueError("generator 80/10/10 策略不匹配")
    if manifest["preprocess"]["selection_count"] != "round(candidate_count*probability)-minimum-one-per-nonempty-row":
        raise ValueError("MLM selection/min-one 策略不匹配")
    if manifest["preprocess"]["seed_derivation"] != expected_seed_derivation45:
        raise ValueError("随机流派生规则不匹配")
    if manifest["preprocess"]["sample_domain"] != ORDINARY45:
        raise ValueError("replacement 采样域不匹配")
    if manifest["preprocess"]["rtd_truth"] != "corrupted_id != original_id":
        raise ValueError("RTD 标签语义不匹配")
    if manifest["preprocess"]["excluded_from_losses"] != sorted(SPECIAL45):
        raise ValueError("special 排除规则不匹配")
    for record in manifest["dataset"]:
        if any(token not in ORDINARY45 for token in record["tokens"]):
            raise ValueError("原始数据含非法 token")
    loaded = ElectraPretrainer45(**manifest["architecture"])
    loaded.load_state_dict(package["state"], strict=True)
    loaded.eval()
    return loaded


loaded45 = load_published_electra45(package45, subject45)
with torch.no_grad():
    assert torch.allclose(
        loaded45.generator_logits(generator_input45, train_attention45),
        model45.generator_logits(generator_input45, train_attention45),
        atol=1e-7,
    )
    assert torch.allclose(
        loaded45.discriminator_logits(train_ids45, train_attention45),
        model45.discriminator_logits(train_ids45, train_attention45),
        atol=1e-7,
    )
assert canonical_state_digest45(state45) == package45["internal"]["state_digest"]

forged45 = copy.deepcopy(package45)
key45 = sorted(forged45["state"])[0]
forged45["state"][key45].view(-1)[0] += 1.0
forged45["internal"]["state_digest"] = canonical_state_digest45(forged45["state"])
try:
    load_published_electra45(forged45, subject45)
    raise AssertionError("重算内部 state hash 的伪造未被拒绝")
except ValueError as error45:
    assert "registry" in str(error45)

replacement45 = copy.deepcopy(package45)
replacement45["manifest"]["preprocess"]["rtd_truth"] = "selected_position"
replacement_key45 = sorted(replacement45["state"])[-1]
replacement45["state"][replacement_key45].view(-1)[-1] -= 0.25
replacement45["internal"]["manifest_digest"] = hashlib.sha256(
    canonical_json45(replacement45["manifest"])
).hexdigest()
replacement45["internal"]["state_digest"] = canonical_state_digest45(
    replacement45["state"]
)
try:
    load_published_electra45(replacement45, subject45)
    raise AssertionError("整体替换并重算内部 hash 未被拒绝")
except ValueError as error45:
    assert "registry" in str(error45)

try:
    PUBLISHER_REGISTRY45[subject45] = "attacker"
    raise AssertionError("registry 不应可写")
except TypeError:
    pass

## 10. 失败模式、生产差距与资料

高频错误包括：把 selected 位置直接标成 replaced；允许采样 MASK/CLS；对 padding 算 BCE；generator logits 未 detach 就让 RTD 反向穿过离散采样；embedding 名义共享但实际复制；切分发生在 masking 之后；只报 discriminator accuracy 却忽略严重类别不平衡。

生产预训练还需要大规模 tokenizer 与动态 masking、数据去重和污染审计、generator/discriminator 容量搜索、分布式混合精度、checkpoint 恢复、吞吐与显存分析。下游价值必须在冻结评测协议和独立数据上验证。

原始资料：

- [ELECTRA: Pre-training Text Encoders as Discriminators Rather Than Generators](https://arxiv.org/abs/2003.10555)
- [Google Research ELECTRA 官方实现](https://github.com/google-research/electra)
- [BERT: Pre-training of Deep Bidirectional Transformers](https://arxiv.org/abs/1810.04805)